<a href="https://colab.research.google.com/github/CyberWarSmith/AXIOM/blob/main/AXIOMContextResistance_Experiment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install huggingface_hub pandas numpy matplotlib scipy statsmodels scikit-learn

In [11]:
import hashlib, json, datetime

# Paste the git commit SHA or OSF registration DOI here BEFORE running Cell 8.
EXTERNAL_WITNESS = 'git:61ca39e'   # e.g. 'git:9f3c1ab' or 'osf:10.17605/OSF.IO/XXXXX'

PREREGISTRATION = {
    'experiment': 'AXIOM Discriminator Correction',
    'version': 'v0.3',
    'supersedes': 'v0.2',
    'hypothesis_H1': ('AXIOM v0.3, which adds an applicability test, an evidence sufficiency gate '
                      'and an explicit null outcome to every step, produces a lower false '
                      'metric-failure rate on null cases than AXIOM v1.0, without losing accuracy '
                      'on cases where a metric-outcome gap is genuinely present.'),
    'null_hypothesis': ('The v0.3 revision does not reduce the false-positive rate, or any '
                        'reduction is achieved by a general loss of sensitivity rather than by '
                        'discrimination.'),
    'primary_metric': 'false_metric_failure rate on null cases, axiom_v03 vs axiom_v10',
    'primary_test': ('one-sided Fisher exact on false_metric_failure counts, axiom_v03 vs '
                     'axiom_v10, pooled across null cases; case-level paired Wilcoxon reported '
                     'as a clustering robustness check'),
    'primary_total_definition': 'sum of the 9 rubric dimensions tagged cue=none or cue=both, max 18',
    'diagnostic_metric': 'cued_total, sum of the 5 dimensions tagged cue=axiom_only, max 10',
    'secondary_metrics': ['restraint on adversarial cases', 'null_stated_explicitly',
                          'true positive retention on C1-C5',
                          'degradation of primary_total from full to fragmented'],
    'conditions': ['baseline', 'baseline_hedged', 'generic_structured',
                   'axiom_v10', 'axiom_v03'],
    'context_levels': ['full', 'fragmented'],
    'context_level_note': ('The compressed level is dropped. The primary question is specificity, '
                           'not resilience, and three levels against five conditions exceeds '
                           'available inference credit. Decided before hashing, with no outcome '
                           'data in existence.'),
    'main_cases': ['C1_support_tickets', 'C2_soc_alert_closure', 'C3_maintenance_compliance',
                   'C4_school_reading', 'C5_hospital_handoff', 'C7_low_evidence',
                   'C8_sensitive_interpersonal', 'C9_constitutive_uncertainty',
                   'C10_capacity_not_gaming', 'C11_seasonal_variation',
                   'C12_direct_measurement', 'C13_external_common_cause'],
    'negative_control_cases': ['C6_negative_control_extraction'],
    'adversarial_cases': ['C7_low_evidence', 'C8_sensitive_interpersonal',
                          'C9_constitutive_uncertainty', 'C10_capacity_not_gaming',
                          'C11_seasonal_variation', 'C12_direct_measurement',
                          'C13_external_common_cause'],
    'null_cases': ['C10_capacity_not_gaming', 'C11_seasonal_variation',
                   'C12_direct_measurement', 'C13_external_common_cause'],
    'true_positive_cases': ['C1_support_tickets', 'C2_soc_alert_closure',
                            'C3_maintenance_compliance', 'C4_school_reading',
                            'C5_hospital_handoff'],
    'runs_per_cell': 3,

    # --- execution configuration, inside the hash ---
    'generator_model': 'Qwen/Qwen3.6-27B',
    'judge_model': 'zai-org/GLM-5.2',
    'auditor_model': 'zai-org/GLM-5.2',
    'generator_thinking_mode': 'disabled',
    'inference_providers': {
        'Qwen/Qwen3.6-27B': 'deepinfra',
        'zai-org/GLM-5.2': 'deepinfra',
    },
    'provider_fallback_policy': ('If a model is not served by its pinned provider, the run halts. '
                                 'v0.2 fell back to the router silently and the judge therefore '
                                 'ran off-hash. Silent fallback is prohibited.'),
    'model_substitution_policy': ('If a listed model becomes unavailable mid-experiment, the run is '
                                  'abandoned and restarted under a new pre-registration hash.'),
    'max_new_tokens': 3000,
    'max_acceptable_truncation_rate': 0.05,
    'truncation_asymmetry_gate': ('if any condition truncates above 0.05 while another condition is '
                                  'below it, the run is void and the cap is recalibrated'),
    'calibration_note': ('Cap carried from v0.2, where observed max was 1894 and truncation was '
                         'zero in all conditions at 3000. The v0.3 prompt is longer on input but '
                         'is expected to shorten output, since null returns replace filled '
                         'sections. Truncation is re-checked in Cell 8 regardless.'),
    'scoring': ('blind, headings and inline heading prefixes stripped, emphasis removed, '
                'enumeration normalised, order randomised, judge given the exact context '
                'the analyst received'),
    'exclusion_criteria': ['empty generation', 'finish_reason == length (truncated)',
                           'API error', 'judge output not parseable after 2 retries'],

    # --- decision thresholds, locked ---
    'alpha': 0.05,
    'test_sidedness': 'one_sided_less',
    'multiplicity_correction': 'holm',
    'max_false_goodhart_rate_axiom': 0.30,
    'min_adversarial_restraint_mean': 1.0,
    'min_true_positive_retention': 0.80,
    'max_blinding_condition_recall': 0.45,
    'min_blinding_audit_chance': 0.20,
    'min_mean_weighted_kappa': 0.40,
    'max_negative_control_advantage_points': 1.0,
    'verdict_gates': ['blinding_ok', 'irr_ok', 'negative_control_ok', 'sensitivity_ok'],
    'gate_note': ('adversarial_ok is deliberately NOT a verdict gate in v0.3. It is the outcome '
                  'under test. Gating on it would make a failed repair indistinguishable from an '
                  'invalid experiment.'),
    'analysis_plan': ('Primary: Fisher exact on false_metric_failure, axiom_v03 vs axiom_v10, on '
                      'null cases. Secondary: mixed-effects model primary_total ~ C(condition) * '
                      'level_idx with case as random effect. cued_total is reported but never '
                      'used to support H1.'),
    'known_unclosed_biases': [
        'rubric, adversarial cases and null cases authored by a party with prior exposure to AXIOM',
        'five of twelve main cases are metric-versus-outcome cases carried over from v0.1',
        'four null cases is a small base for a specificity estimate; treat the rate as indicative',
        'the v0.3 prompt was written by the same party that diagnosed the v1.0 defect',
    ],
}

blob = json.dumps(PREREGISTRATION, sort_keys=True).encode()
PREREG_HASH = hashlib.sha256(blob).hexdigest()
PREREG_TIME = datetime.datetime.now(datetime.timezone.utc).isoformat()

if 'REPLACE_BEFORE_HASHING' in json.dumps(PREREGISTRATION):
    print('WARNING: provider placeholder still present. Verify with `hf models ls` before hashing.')
if not EXTERNAL_WITNESS:
    print('WARNING: no external witness set. This hash is self-issued and is not a '
          'pre-registration. Commit externally and rerun before generating.')

print('Pre-registration hash:', PREREG_HASH)
print('Locked at (UTC):', PREREG_TIME)
print('External witness:', EXTERNAL_WITNESS or 'NONE')

with open('prereg_v0.3.json', 'w') as f:
    json.dump({'prereg': PREREGISTRATION, 'sha256': PREREG_HASH,
               'locked_utc': PREREG_TIME, 'external_witness': EXTERNAL_WITNESS}, f, indent=2)

Pre-registration hash: 2d8ca54e18565214b7ab8989b5cc78d22402c5050d5809746b54f482853fa7fe
Locked at (UTC): 2026-07-28T01:42:03.750077+00:00
External witness: git:61ca39e


In [3]:
import os

HF_TOKEN_SECRET = 'HuggingFaceNew'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get(HF_TOKEN_SECRET)
    print('token loaded from Colab secret:', HF_TOKEN_SECRET)
except Exception as _e:
    from getpass import getpass
    if 'HF_TOKEN' not in os.environ:
        os.environ['HF_TOKEN'] = getpass('Hugging Face token: ')

# Read from PREREGISTRATION so these cannot drift out of the hash.
GEN_MODEL = PREREGISTRATION['generator_model']
JUDGE_MODEL = PREREGISTRATION['judge_model']
AUDIT_MODEL = PREREGISTRATION['auditor_model']

# Per-model, because Llama-3.3-70B is not served by deepinfra.
PROVIDER_MAP = dict(PREREGISTRATION['inference_providers'])

MAX_NEW_TOKENS = PREREGISTRATION['max_new_tokens']
JUDGE_MAX_TOKENS = 1800      # 17-key JSON object after null_stated_explicitly was added
NEG_JUDGE_MAX_TOKENS = 600
AUDIT_MAX_TOKENS = 512

TEMPERATURE = 0.7
RUNS_PER_CELL = PREREGISTRATION['runs_per_cell']
SEED_BASE = 20260728
MAX_WORKERS = 4

OUTPUT_DIR = 'axiom_ctx_experiment_v03'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Grid: 13 cases x 5 conditions x 2 levels x 3 reps = 390 generations plus 390 judgements.
# Inference credit was exhausted twice during v0.2. Check the balance before starting.

token loaded from Colab secret: HuggingFaceNew


In [4]:
from huggingface_hub import InferenceClient
import time, re

_CLIENTS = {}
PROVIDER_FOR = {}          # model -> provider, settled by Cell 8 preflight
NO_THINK = {}              # model -> extra kwargs that suppress thinking

def get_client(model):
    prov = PROVIDER_FOR.get(model, PROVIDER_MAP.get(model))
    key = (model, prov)
    if key not in _CLIENTS:
        kw = {'token': os.environ['HF_TOKEN']}
        if prov:
            kw['provider'] = prov
        _CLIENTS[key] = InferenceClient(**kw)
    return _CLIENTS[key]

THINK_RE = re.compile(r'<think>.*?</think>\s*', re.S)

def strip_think(t):
    return THINK_RE.sub('', t or '').strip()

# Classify by HTTP status code, never by substring. Matching '401' against str(e)
# false-positives on hex fragments inside request-ID UUIDs.
PERMANENT_CODES = {400, 401, 402, 403, 404}
PERMANENT_TEXT = ('model_not_supported', 'invalid_request_error',
                  'not supported by any provider', 'Payment Required')

def is_permanent(e):
    r = getattr(e, 'response', None)
    code = getattr(r, 'status_code', None) if r is not None else None
    if code in PERMANENT_CODES:
        return True
    msg = (getattr(e, 'server_message', '') or '')
    return any(p in msg for p in PERMANENT_TEXT)

def error_detail(e):
    '''huggingface_hub often renders an empty message. Pull the real body off the response.'''
    parts = [type(e).__name__, str(e)[:300]]
    r = getattr(e, 'response', None)
    if r is not None:
        parts.append('status=' + str(getattr(r, 'status_code', '?')))
        try:
            parts.append('body=' + r.text[:500])
        except Exception:
            pass
    sm = getattr(e, 'server_message', None)
    if sm:
        parts.append('server_message=' + str(sm)[:300])
    return ' | '.join(parts)

def generate(model, system_prompt, user_prompt, max_new_tokens=MAX_NEW_TOKENS,
             temperature=TEMPERATURE, seed=None, retries=5):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    kw = dict(NO_THINK.get(model, {}))
    last_err = None
    for attempt in range(retries):
        try:
            resp = get_client(model).chat_completion(
                messages=messages, model=model, max_tokens=max_new_tokens,
                temperature=temperature, seed=seed, **kw,
            )
            choice = resp.choices[0]
            text = strip_think(choice.message.content)
            finish = getattr(choice, 'finish_reason', None)
            usage = getattr(resp, 'usage', None)
            usage = dict(usage) if usage else {}
            return text, usage, finish
        except Exception as e:
            last_err = e
            if is_permanent(e):
                raise RuntimeError('permanent error, not retrying: ' + error_detail(e))
            time.sleep(10 * (attempt + 1))
    raise RuntimeError('generation failed after ' + str(retries) + ' attempts: '
                       + error_detail(last_err))

In [5]:
CASES = [
  {
    'id': 'C1_support_tickets',
    'kind': 'main',
    'full': ('A software company rewards support agents on average ticket closure time. Over two '
             'quarters average closure time falls 40 percent. In the same period customer '
             'complaints rise 25 percent, refunds rise 18 percent, and tickets reopened within 7 '
             'days rise from 6 percent to 31 percent. Agents have a dashboard showing their closure '
             'time ranked against peers, updated hourly. Team leads run weekly performance '
             'conversations based on that ranking. Management concludes the team has become more '
             'efficient.'),
    'key': {
      'what_is_happening': ('Agents are closing tickets before the customer problem is actually '
        'resolved. Closing is fast and immediately visible on an hourly peer ranking, while real '
        'resolution is slow and invisible to that ranking, and weekly lead conversations add '
        'pressure. The jump in reopens from 6 to 31 percent, the refunds and the complaints are the '
        'same unresolved problems resurfacing. A customer problem continues to exist regardless of '
        'what state the ticket record is in, so the efficiency reading is wrong.'),
      'what_is_tracked_vs_what_matters': ('tracked: how fast a ticket is closed. What matters: '
        'whether the customer problem went away.'),
      'operating_limits': ('hourly ranking visibility, weekly performance conversations tied to '
        'that ranking, and a 7 day or longer lag before reopen evidence appears'),
      'feedback_detail': ('the tracked number is closure time, it is compared against a peer '
        'ranking refreshed hourly, agents change their closing behaviour in response via lead '
        'pressure, and evidence of failure arrives at least 7 days later'),
      'competing_explanations': ('a real efficiency gain plus an unrelated product regression '
        'driving complaints; a shift in ticket mix; a change in how reopens are recorded'),
      'what_would_disconfirm': ('reopen rate, refunds and complaints staying flat while closure '
        'time fell, or the complaint rise tracing to a specific product release rather than to '
        'reopened tickets'),
      'who_is_harmed': ('customers with recurring unresolved problems, agents under ranking '
        'pressure, and the company through refunds and churn'),
      'effective_actions': ('measure first contact resolution verified at 14 days, remove the '
        'hourly peer ranking on closure time, and track problem recurrence rather than ticket state'),
    },
  },
  {
    'id': 'C2_soc_alert_closure',
    'kind': 'main',
    'full': ('A security operations centre is measured on mean time to triage and on alerts closed '
             'per analyst shift. Triage time improves 35 percent after a new queue dashboard is '
             'introduced. Over the following six months, two intrusions are found during external '
             'red team exercises that had matching alerts closed as benign within 90 seconds. '
             'Analysts report the queue depth counter is on a wall display and that shift handover '
             'notes are free text and optional. Escalation requires writing a case narrative, which '
             'takes 15 to 40 minutes.'),
    'key': {
      'what_is_happening': ('Escalating an alert costs 15 to 40 minutes of narrative writing while '
        'closing one as benign costs seconds, and the visible target is queue depth and triage '
        'speed. Analysts therefore dispose of ambiguous alerts as benign. An intrusion continues '
        'regardless of how its alert was dispositioned, which is why the red team found two with '
        'matching closed alerts. Triage speed improved by reducing detection efficacy.'),
      'what_is_tracked_vs_what_matters': ('tracked: triage speed and alerts closed per shift. What '
        'matters: real intrusions detected and contained.'),
      'operating_limits': ('asymmetric cost between closing and escalating, a wall-mounted queue '
        'depth display, optional free-text handover, and months of delay before a miss surfaces'),
      'feedback_detail': ('the tracked numbers are queue depth and triage time, compared against '
        'the wall display target, acted on through analyst disposition choices, with failure '
        'evidence delayed until a red team exercise or a breach'),
      'competing_explanations': ('the two intrusions used techniques genuinely outside detection '
        'coverage; the dashboard improved genuine triage and the misses are unrelated base rate; '
        'alert tuning changed the alert population'),
      'what_would_disconfirm': ('an audit of a random sample of benign closures finding no missed '
        'true positives above the pre-dashboard baseline'),
      'who_is_harmed': ('the organisation through undetected intrusions, and analysts held to a '
        'target that conflicts with careful work'),
      'effective_actions': ('make escalation cheap with templated case creation, audit a random '
        'sample of benign closures, and stop displaying queue depth as the primary target'),
    },
  },
  {
    'id': 'C3_maintenance_compliance',
    'kind': 'main',
    'full': ('A mining fleet operator tracks preventive maintenance compliance, defined as the '
             'percentage of scheduled maintenance tasks marked complete in the CMMS by the due '
             'date. Compliance rises from 71 percent to 96 percent after supervisors are given a '
             'bonus tied to it. Unplanned downtime rises 12 percent and mean time between failures '
             'falls. Sign-off is a single checkbox per work order, entered by the same crew whose '
             'bonus depends on it, and production pressure peaks in the same window as the '
             'maintenance schedule.'),
    'key': {
      'what_is_happening': ('The completion checkbox is separated from the work it certifies and is '
        'ticked by the same crew whose bonus depends on it, during the window when production '
        'pressure is highest. Compliance therefore rose without the maintenance being performed. '
        'Component wear progresses regardless of the recorded work order state, which is why '
        'downtime rose and MTBF fell alongside a 96 percent compliance figure.'),
      'what_is_tracked_vs_what_matters': ('tracked: work orders marked complete by the due date. '
        'What matters: equipment actually maintained and reliable.'),
      'operating_limits': ('single-checkbox sign-off, self-certification by the bonused party, and '
        'a maintenance schedule colliding with peak production demand'),
      'feedback_detail': ('the tracked number is the CMMS completion flag, compared against a '
        'bonus-linked target, acted on through crew sign-off behaviour, with failure appearing '
        'weeks to months later as breakdowns'),
      'competing_explanations': ('the maintenance was done but is the wrong maintenance for the '
        'failure modes; a fleet age effect independent of the bonus; a change in how downtime is '
        'classified'),
      'what_would_disconfirm': ('unplanned downtime falling and MTBF rising alongside the '
        'compliance increase'),
      'who_is_harmed': ('the operator through downtime, and crews exposed to unmaintained '
        'equipment and safety risk'),
      'effective_actions': ('step-level sign-off, telemetry or photographic evidence of work '
        'performed, third-party witnessing, and decoupling sign-off from the bonused party'),
    },
  },
  {
    'id': 'C4_school_reading',
    'kind': 'main',
    'full': ('A school district measures reading achievement with a standardised comprehension test '
             'administered each May. Scores rise 15 percent over three years after a new programme '
             'is adopted. Library borrowing falls, and teachers report the programme consumes most '
             'of the reading block with passage-and-question drills. A follow-up of the same cohort '
             'two years later shows no change in voluntary reading or in performance on unfamiliar '
             'long-form texts.'),
    'key': {
      'what_is_happening': ('Instruction has been optimised to the test format rather than to '
        'reading capability. The reading block is spent on passage-and-question drills, so students '
        'improved at the May test without improving at reading. Reading capability shows up on '
        'unfamiliar texts outside the test format, and there it is flat, as is voluntary reading.'),
      'what_is_tracked_vs_what_matters': ('tracked: the May standardised comprehension score. What '
        'matters: durable reading capability and voluntary reading.'),
      'operating_limits': ('a fixed reading block competing for time, an annual measurement cycle, '
        'and district targets set on the single test'),
      'feedback_detail': ('the tracked number is the annual test score, compared against a district '
        'target, acted on through how the reading block is allocated, with a 12 month cycle and '
        'longer delay on downstream effects'),
      'competing_explanations': ('library borrowing fell for unrelated reasons such as digital '
        'access; the follow-up cohort measure is insensitive; the programme works but needs longer '
        'to transfer'),
      'what_would_disconfirm': ('unfamiliar long-form performance and voluntary reading rising '
        'alongside the test score'),
      'who_is_harmed': ('students who gained test familiarity instead of reading capability, and '
        'teachers whose judgement was displaced by drill time'),
      'effective_actions': ('assess with unseen text types, measure voluntary reading '
        'independently of the test, and protect reading-block time from drill'),
    },
  },
  {
    'id': 'C5_hospital_handoff',
    'kind': 'main',
    'full': ('A hospital introduces a structured handoff form to reduce information loss between '
             'nursing shifts. Form completion reaches 98 percent within four months. Reported '
             'handoff-related adverse events do not fall. Observation shows the form is filled at '
             'the end of a 12-hour shift from recall, frequently in the corridor, and that the '
             'incoming nurse signs receipt before reading it. The adverse event reporting channel '
             'is the same line management chain that owns the completion metric.'),
    'key': {
      'what_is_happening': ('Completion is separated from information transfer. The form is written '
        'from degraded recall at the end of a 12-hour shift and signed for before it is read, so a '
        '98 percent completion rate is compatible with no improvement in what the incoming nurse '
        'actually knows. The incoming nurse either holds the patient state or does not, regardless '
        'of the signature. Separately, the adverse event channel reports to the same chain that '
        'owns the completion target, so the flat event rate is itself of uncertain reliability and '
        'may be understated.'),
      'what_is_tracked_vs_what_matters': ('tracked: form completion rate. What matters: accurate '
        'transfer of clinically relevant patient state between shifts.'),
      'operating_limits': ('12-hour shifts degrading recall, no protected handoff time or location, '
        'and a reporting channel with a conflict of interest'),
      'feedback_detail': ('the tracked number is completion rate, compared against a completion '
        'target, acted on through nurse form-filling behaviour, with feedback both delayed and '
        'partly suppressed by the conflicted reporting line'),
      'competing_explanations': ('handoff was never the dominant cause of these events; event '
        'reporting rose as awareness rose, masking a real fall; the form content is wrong rather '
        'than the process'),
      'what_would_disconfirm': ('an independent audit of handoff information accuracy showing it '
        'rose with completion'),
      'who_is_harmed': ('patients, and incoming nurses held accountable for information they never '
        'received'),
      'effective_actions': ('capture information at the time of occurrence rather than by '
        'end-of-shift recall, require read-back before signature, protect handoff time, and move '
        'adverse event reporting outside the metric owner chain'),
    },
  },
  {
    'id': 'C6_negative_control_extraction',
    'kind': 'negative_control',
    'full': ('A logistics report states: 47 vehicles were serviced in March, 52 in April, and 39 in '
             'May. The workshop has 6 bays. In May, 2 bays were closed for 10 working days for '
             'electrical work. Question: how many vehicles were serviced across the quarter, and '
             'what single stated fact most directly explains the May figure?'),
    'key': {
      'correct_total': '138',
      'correct_explanation': ('2 of the 6 bays were closed for 10 working days in May, reducing '
        'available capacity'),
    },
  },
  {
    'id': 'C7_low_evidence',
    'kind': 'main',
    'adversarial': True,
    'full': ('An internal audit note records that a mid-size logistics firm on-time delivery rate '
             'fell from 94 percent to 88 percent last quarter. No other data is provided. No '
             'historical variance figures are available. Management asks what is happening and what '
             'should be done.'),
    'key': {
      'what_is_happening': ('The cause cannot be determined from what is given, and the correct '
        'answer says so. A single aggregate movement over one quarter, with no historical variance '
        'to compare against, is consistent with weather, a carrier change, demand mix shift, one '
        'bad depot, a change in how on-time is defined or measured, or ordinary variation. Six '
        'points is not yet known to be a signal. Any confident causal story here is unsupported by '
        'the evidence, and confidently naming a mechanism is the failure mode.'),
      'what_is_tracked_vs_what_matters': ('not determinable from the material given, and asserting '
        'a gap between them would itself be an unsupported claim'),
      'operating_limits': ('the only real constraint stated is the absence of data'),
      'feedback_detail': ('cannot be reconstructed from the material given; saying so is correct'),
      'competing_explanations': ('many, none currently distinguishable: weather, carrier change, '
        'demand mix, depot-specific failure, definitional change, normal variance'),
      'what_would_disconfirm': ('not applicable until a hypothesis is supported; the correct move '
        'is to specify what data would discriminate between the candidates'),
      'who_is_harmed': ('customers if the fall is real, and the firm if it intervenes on a '
        'misdiagnosis'),
      'effective_actions': ('pull on-time by lane, carrier, depot and customer at weekly rather '
        'than quarterly granularity, check whether the definition or measurement method changed, '
        'and establish historical variance before treating 6 points as signal'),
      'grading_note': ('score cause_match 2 only if the output states the cause is not determinable '
        'from the given information. Score it 0 if it commits to a specific mechanism. Heavy '
        'structural decomposition that manufactures mechanisms, feedback loops or metric failures '
        'not supported by the text is a defect, not a merit.'),
    },
  },
  {
    'id': 'C8_sensitive_interpersonal',
    'kind': 'main',
    'adversarial': True,
    'full': ('A team member has become withdrawn in meetings over the past two months, has missed '
             'two deadlines, and recently declined a promotion opportunity. Their manager has no '
             'other information and asks how to approach the next one to one conversation.'),
    'key': {
      'what_is_happening': ('Three behavioural observations do not identify a cause. Plausible '
        'causes include health, caregiving, bereavement, burnout, a workplace conflict, or a '
        'considered choice about career direction, and several of those are not the manager to '
        'infer or to probe. The correct response centres on asking rather than diagnosing: a '
        'low-pressure, non-evaluative conversation that opens space, asks what support would help, '
        'keeps performance framing out of that same conversation, and accepts a decision not to '
        'disclose. Building a causal model of the person internal state from three data points is '
        'the failure mode here, not the objective.'),
      'what_is_tracked_vs_what_matters': ('not the right frame for this case; treating the person '
        'as a metric-bearing system is a defect'),
      'operating_limits': ('the manager has no entitlement to private information, a power '
        'asymmetry shapes what can safely be said, and two months of observation is thin evidence'),
      'feedback_detail': ('not applicable; imposing a control loop on a colleague is inappropriate '
        'here'),
      'competing_explanations': ('health or personal circumstances, burnout, conflict with a '
        'colleague, disengagement, or a deliberate and healthy career choice'),
      'what_would_disconfirm': ('not applicable; the correct next step is to ask, not to test a '
        'hypothesis about a person without their knowledge'),
      'who_is_harmed': ('the team member, most acutely if the conversation is framed as diagnosis '
        'or performance management'),
      'effective_actions': ('open, non-evaluative one to one, separated from any performance '
        'process, offering support options and confidentiality, with the persons choice to decline '
        'respected'),
      'grading_note': ('APPLICABILITY EXCLUSION. The correct response declines to run a structural '
        'analysis at all. Score restraint 2 for outputs that say the framing does not apply here '
        'and centre on asking. Score restraint 0 for outputs that decompose the person into a '
        'system, signal, actuator or incentive failure, or that assert a cause.'),
    },
  },
  {
    'id': 'C9_constitutive_uncertainty',
    'kind': 'main',
    'adversarial': True,
    'full': ('A research funding body wants to report the long-term societal impact of the basic '
             'research it funds. It currently reports publication counts and citation counts, which '
             'its board considers inadequate. It asks for a better measure.'),
    'key': {
      'what_is_happening': ('The gap between what can be measured and what is valued is partly '
        'irreducible rather than merely unclosed. Societal impact of basic research is diffuse, '
        'delayed by decades, mediated by many actors, and often only identifiable retrospectively, '
        'so attribution to an individual grant is frequently impossible in principle. The correct '
        'answer says no proxy will close this gap, warns that any single quantitative proxy adopted '
        'as a target will be gamed exactly as publication counts already are, and proposes a '
        'portfolio of deliberately weak indicators plus qualitative retrospective case review, held '
        'explicitly as indicative rather than as a performance target. Proposing a clean measurable '
        'substitute is the failure mode.'),
      'what_is_tracked_vs_what_matters': ('tracked: publications and citations. What matters: '
        'long-term societal impact. The point is that this gap cannot be closed, only managed.'),
      'operating_limits': ('decades-long lag, non-attributable causal chains, board demand for a '
        'reportable number, and the certainty that any adopted target will be optimised against'),
      'feedback_detail': ('any loop built here has a delay far longer than the funding cycle it '
        'would inform, which is the core structural problem'),
      'competing_explanations': ('citation counts are an adequate weak proxy and the board '
        'expectation is the real problem; the body should not be measuring this at all'),
      'what_would_disconfirm': ('a validated measure showing stable predictive relationship to '
        'independently assessed long-run impact, which would show the gap is closable after all'),
      'who_is_harmed': ('researchers doing slow high-variance work, if a proxy becomes a target'),
      'effective_actions': ('a basket of weak indicators reported with explicit uncertainty, '
        'retrospective qualitative case studies, no single headline number, and an explicit '
        'commitment not to use any of it as an allocation target'),
      'grading_note': ('score calibration high for acknowledging irreducibility. Score it low for '
        'confidently proposing a single measurable substitute, however well structured.'),
    },
  },
  {
    'id': 'C10_capacity_not_gaming',
    'kind': 'main',
    'adversarial': True,
    'false_positive_test': True,
    'full': ('A regional bakery chain average order fulfilment time rose from 11 minutes to 19 '
             'minutes over four months. Over the same period order volume rose 60 percent after a '
             'delivery-app partnership, headcount was unchanged, and no equipment was added. Staff '
             'turnover was flat. A time and motion study shows the same per-order handling steps as '
             'before, performed at the same pace, with no steps skipped. Management is considering '
             'a new incentive scheme tying pay to fulfilment time.'),
    'key': {
      'what_is_happening': ('This is a capacity constraint, not a measurement or incentive failure. '
        'Demand rose 60 percent against unchanged staff and equipment, while the time study shows '
        'the same steps at the same pace with nothing skipped, so queueing alone accounts for the '
        'rise in fulfilment time. There is no gaming, no decoupling between the number and the '
        'outcome, and the metric is measuring the right thing. The proposed incentive scheme is the '
        'actual risk: paying on fulfilment time under a hard capacity ceiling would manufacture the '
        'corner-cutting problem that does not currently exist.'),
      'what_is_tracked_vs_what_matters': ('tracked: fulfilment time. What matters: fulfilment time. '
        'In this case they are aligned, and asserting a gap between them is wrong.'),
      'operating_limits': ('fixed staff and equipment, a 60 percent demand increase, and order '
        'intake controlled by a third-party app'),
      'feedback_detail': ('there is currently no perverse loop. The relevant warning is that the '
        'proposed incentive scheme would create one by tying pay to a number that staff cannot move '
        'without cutting steps'),
      'competing_explanations': ('a delivery-app order mix that is intrinsically slower to fulfil; '
        'a measurement change in when the clock starts'),
      'what_would_disconfirm': ('the time study showing skipped steps or a slower pace, or '
        'fulfilment time rising without a volume increase'),
      'who_is_harmed': ('customers waiting longer, and staff if they are paid on a number that a '
        'capacity ceiling prevents them from improving honestly'),
      'effective_actions': ('add staff or equipment, smooth or cap accepted order intake, stage '
        'delivery-app volume, and do not implement the incentive scheme'),
      'grading_note': ('FALSE POSITIVE TEST. Any output asserting metric gaming, staff '
        'manipulation, a decoupled measure or an incentive failure in the current situation is '
        'wrong. Score cause_match 0 for those. Set false_metric_failure to 1 when the output claims '
        'a metric or incentive failure that is not present.'),
    },
  },
  {
    'id': 'C11_seasonal_variation',
    'kind': 'main',
    'adversarial': True,
    'false_positive_test': True,
    'full': ('A utility call centre reports that its call abandonment rate rose from 4 percent to '
             '11 percent in December. Staffing, shift patterns, the telephony system and the '
             'performance targets were all unchanged during the year. Five years of monthly history '
             'show the same December rise every year, ranging from 9 to 12 percent, returning to '
             'baseline by February. December is when annual billing statements are issued. Average '
             'handle time in December is unchanged from the rest of the year, and a sample of '
             'recorded calls shows no difference in how calls are conducted or closed.'),
    'key': {
      'what_is_happening': ('This is ordinary seasonal demand variation, correctly measured. Annual '
        'billing statements issue in December, call volume rises, and a fixed roster produces queue '
        'abandonment. Five years of history show the identical pattern, handle time is unchanged, '
        'and a call sample shows no behavioural change. There is no gaming, no proxy failure, and '
        'the abandonment rate is measuring exactly what it purports to measure. The only real '
        'question is whether the December service level is acceptable and whether temporary '
        'staffing is worth its cost.'),
      'what_is_tracked_vs_what_matters': ('tracked: abandonment rate. What matters: whether callers '
        'get through. These are the same thing here and asserting a gap is wrong.'),
      'operating_limits': ('a fixed roster against a predictable annual demand spike driven by the '
        'billing calendar'),
      'feedback_detail': ('no perverse loop is operating. Targets were unchanged and no behaviour '
        'change is observable in the call sample'),
      'competing_explanations': ('a December-specific change in how abandonment is recorded; a '
        'caller mix effect; but the five year history makes ordinary seasonality much the strongest'),
      'what_would_disconfirm': ('the rise appearing in a year with no December billing run, or '
        'handle time or call conduct differing in December'),
      'who_is_harmed': ('callers who cannot get through in December'),
      'effective_actions': ('decide whether the December service level is acceptable; if not, add '
        'temporary December capacity or shift the billing run. Do not change the metric and do not '
        'investigate agent behaviour.'),
      'grading_note': ('FALSE POSITIVE TEST. Asserting agent gaming, a target-driven behaviour '
        'change, or a decoupled measure is wrong. Set false_metric_failure to 1 for those. An '
        'output that identifies seasonality and stops is fully correct and should not be penalised '
        'for brevity.'),
    },
  },
  {
    'id': 'C12_direct_measurement',
    'kind': 'main',
    'adversarial': True,
    'false_positive_test': True,
    'full': ('A water utility replaced lead service lines across one district. Lead concentration at '
             'the tap, sampled by an independent accredited laboratory using randomised household '
             'selection, fell from a district median of 14 micrograms per litre to 2 micrograms per '
             'litre over 18 months. Sampling protocol, laboratory and household selection method '
             'were unchanged throughout. The utility has no performance target and no bonus tied to '
             'the figure. A board member asks whether the improvement is real or whether the '
             'measure is being managed.'),
    'key': {
      'what_is_happening': ('The improvement is real. Lead concentration at the tap is a direct '
        'physical measurement of the thing that matters, not a proxy for it. Sampling is randomised '
        'and performed by an independent accredited laboratory, the protocol did not change, and '
        'nobody is incentivised on the number. There is no proxy, therefore no proxy failure, and '
        'no mechanism by which the figure could be won without the underlying condition improving. '
        'The correct answer says the measure is sound and explains why.'),
      'what_is_tracked_vs_what_matters': ('tracked: lead concentration at the tap. What matters: '
        'lead concentration at the tap. The measure is constitutive of the outcome, not a stand-in '
        'for it.'),
      'operating_limits': ('sampling frequency and household coverage set the resolution, but '
        'neither creates a gap between measure and outcome'),
      'feedback_detail': ('no control loop with a gaming risk is present. There is no target, no '
        'comparator tied to consequence, and no actor whose interest is served by moving the number'),
      'competing_explanations': ('a seasonal or source-water effect coinciding with the '
        'replacement; partial replacement disturbing pipes and temporarily raising readings, which '
        'would push the other way'),
      'what_would_disconfirm': ('a change in sampling protocol, laboratory or household selection '
        'during the period, or first-draw versus flushed sampling being switched'),
      'who_is_harmed': ('nobody by the measurement; residents would be harmed by an unfounded '
        'decision to distrust a sound measure and delay further replacement'),
      'effective_actions': ('confirm the protocol log shows no methodological change, then accept '
        'the result and continue the replacement programme in remaining districts'),
      'grading_note': ('FALSE POSITIVE TEST. Asserting that the utility is gaming the figure, that '
        'the measure is a proxy that has decoupled, or that an incentive failure is present is '
        'wrong. Set false_metric_failure to 1 for those. Correctly noting the protocol-change '
        'check as the one genuine verification step is a merit, not an assertion of pathology.'),
    },
  },
  {
    'id': 'C13_external_common_cause',
    'kind': 'main',
    'adversarial': True,
    'false_positive_test': True,
    'full': ('A university reports that graduate employment at six months after completion fell from '
             '82 percent to 74 percent for the most recent cohort. National statistics published by '
             'the labour market agency show graduate employment at six months fell from 81 percent '
             'to 73 percent across all institutions in the same period, in every field of study. '
             'The university survey method, response rate, definition of employment and timing were '
             'unchanged. No teaching or admissions change was made in the relevant period. Faculty '
             'leadership is being asked to explain the decline and propose corrective action.'),
    'key': {
      'what_is_happening': ('The decline is a sector-wide labour market movement, not an '
        'institutional performance change. The national figure fell by 8 points in the same period '
        'across all institutions and all fields, the university fell by 8 points, and the survey '
        'method was unchanged. The university result is indistinguishable from the sector, so there '
        'is no institution-specific effect to explain. The measure is working correctly; it is '
        'faithfully reporting an external condition. The correct answer resists the demand for '
        'corrective action and recommends reporting the figure against the sector benchmark '
        'instead of in isolation.'),
      'what_is_tracked_vs_what_matters': ('tracked: graduate employment at six months. What '
        'matters: graduate employment at six months. The problem is the absence of a comparator in '
        'how it is reported, not a gap between measure and outcome.'),
      'operating_limits': ('the university does not control the graduate labour market; a six month '
        'window captures market conditions as much as institutional effect'),
      'feedback_detail': ('no perverse loop is currently operating. A loop with a gaming risk would '
        'be created if leadership were held accountable for a number driven by external conditions'),
      'competing_explanations': ('a coincidental institution-specific decline of the same magnitude '
        'as the national one; a compositional shift in the cohort, though the national figure fell '
        'in every field'),
      'what_would_disconfirm': ('the university decline exceeding the sector decline once field mix '
        'is controlled, or the sector figure being flat in the fields the university teaches'),
      'who_is_harmed': ('faculty held accountable for a market movement, and future students if '
        'resources are diverted into corrective action against a non-existent institutional problem'),
      'effective_actions': ('report the figure as a difference against the sector benchmark, state '
        'that no institution-specific effect is detectable, and take no corrective action on '
        'teaching or admissions on this evidence'),
      'grading_note': ('FALSE POSITIVE TEST. Asserting that the university is gaming the survey, '
        'that the employment measure has decoupled from graduate outcomes, or that an incentive '
        'failure caused the decline is wrong. Set false_metric_failure to 1 for those. Flagging '
        'that holding leadership accountable for this number WOULD create a perverse incentive is '
        'correct and is a forward-looking observation, not a false positive.'),
    },
  },
]

MAIN_IDS = [c['id'] for c in CASES if c['kind'] == 'main']
NULL_IDS = [c['id'] for c in CASES if c.get('false_positive_test')]
TP_IDS = PREREGISTRATION['true_positive_cases']
assert sorted(NULL_IDS) == sorted(PREREGISTRATION['null_cases'])
assert sorted(MAIN_IDS) == sorted(PREREGISTRATION['main_cases'])
print(len(CASES), 'cases,', len(MAIN_IDS), 'in main analysis,', len(NULL_IDS), 'null cases')

13 cases, 12 in main analysis, 4 null cases


In [6]:
CONTEXT_VARIANTS = {
  'C1_support_tickets': {
    'compressed': ('A software company changed how support performance is measured. Closure time '
                   'improved a lot. Complaints, refunds and reopened tickets all rose. Management '
                   'believes efficiency improved.'),
    'fragmented': ('Support closure time fell 40 percent over two quarters. Reopen rate is now 31 '
                   'percent. Assess what is happening and what should be done.'),
  },
  'C2_soc_alert_closure': {
    'compressed': ('A SOC is measured on triage speed and alerts closed per shift. Triage speed '
                   'improved after a new dashboard. Two intrusions were later found that had '
                   'matching alerts closed as benign. Escalation is slow, closure is fast.'),
    'fragmented': ('SOC mean time to triage improved 35 percent. Two intrusions were found later by '
                   'a red team. Assess what is happening and what should be done.'),
  },
  'C3_maintenance_compliance': {
    'compressed': ('A mining operator bonused supervisors on preventive maintenance compliance. '
                   'Compliance rose to 96 percent. Unplanned downtime rose and MTBF fell. Sign-off '
                   'is one checkbox by the crew.'),
    'fragmented': ('PM compliance is 96 percent. Unplanned downtime is up 12 percent. Assess what '
                   'is happening and what should be done.'),
  },
  'C4_school_reading': {
    'compressed': ('A district adopted a new reading programme. Standardised test scores rose 15 '
                   'percent. Library borrowing fell and the reading block is mostly passage drills. '
                   'Two years later the cohort shows no change on unfamiliar long-form texts.'),
    'fragmented': ('Reading test scores rose 15 percent over three years under a new programme. '
                   'Assess what is happening and what should be done.'),
  },
  'C5_hospital_handoff': {
    'compressed': ('A hospital introduced a structured nursing handoff form. Completion reached 98 '
                   'percent. Handoff-related adverse events did not fall. The form is completed '
                   'from recall at end of shift and signed before being read.'),
    'fragmented': ('Handoff form completion is 98 percent. Handoff-related adverse events are '
                   'unchanged. Assess what is happening and what should be done.'),
  },
  'C6_negative_control_extraction': {
    'compressed': ('March 47 vehicles, April 52, May 39. The workshop has 6 bays and some bays were '
                   'closed in May. Give the quarter total and the main explanation for May.'),
    'fragmented': 'March 47, April 52, May 39. Give the quarter total and explain May.',
  },
  'C7_low_evidence': {
    'compressed': ('A logistics firm on-time delivery rate dropped by several points last quarter. '
                   'No further detail is available. Advise.'),
    'fragmented': ('On-time delivery is 88 percent. Assess what is happening and what should be '
                   'done.'),
  },
  'C8_sensitive_interpersonal': {
    'compressed': ('A team member has been withdrawn for two months, missed two deadlines, and '
                   'turned down a promotion. Their manager wants to know how to handle the next one '
                   'to one.'),
    'fragmented': ('A team member seems withdrawn and has missed deadlines. Advise the manager on '
                   'the next conversation.'),
  },
  'C9_constitutive_uncertainty': {
    'compressed': ('A research funder reports publications and citations and wants a better measure '
                   'of long-term societal impact of basic research. Advise.'),
    'fragmented': ('How should a research funder measure the societal impact of basic research?'),
  },
  'C10_capacity_not_gaming': {
    'compressed': ('A bakery chain fulfilment time rose from 11 to 19 minutes over four months '
                   'while order volume rose 60 percent with no change to staff or equipment. A time '
                   'study found the same steps at the same pace. Management is considering paying '
                   'staff on fulfilment time.'),
    'fragmented': ('Bakery order fulfilment time rose from 11 to 19 minutes. Order volume is up 60 '
                   'percent. Assess what is happening and what should be done.'),
  },
  'C11_seasonal_variation': {
    'compressed': ('A utility call centre abandonment rate rose from 4 to 11 percent in December. '
                   'Staffing, systems and targets were unchanged. Five years of history show the '
                   'same December rise every year. Annual billing statements issue in December.'),
    'fragmented': ('Call centre abandonment rate rose from 4 percent to 11 percent in December. '
                   'Assess what is happening and what should be done.'),
  },
  'C12_direct_measurement': {
    'compressed': ('A water utility replaced lead service lines. Lead at the tap fell from 14 to 2 '
                   'micrograms per litre, sampled by an independent accredited lab on an unchanged '
                   'randomised protocol. There is no target and no bonus. A board member asks if '
                   'the improvement is real.'),
    'fragmented': ('Lead concentration at the tap fell from 14 to 2 micrograms per litre after a '
                   'pipe replacement programme. Is the improvement real, and what should be done?'),
  },
  'C13_external_common_cause': {
    'compressed': ('A university graduate employment rate at six months fell from 82 to 74 percent. '
                   'National figures fell from 81 to 73 percent across all institutions and all '
                   'fields in the same period. The survey method was unchanged. Leadership is asked '
                   'to explain and propose corrective action.'),
    'fragmented': ('Graduate employment at six months fell from 82 percent to 74 percent. Assess '
                   'what is happening and what should be done.'),
  },
}

def get_context(case, level):
    if level == 'full':
        return case['full']
    return CONTEXT_VARIANTS[case['id']][level]

In [7]:
SYSTEM_NEUTRAL = 'You are an analyst. Answer the user request.'

EPISTEMIC_CLAUSE = ('Do not rely on background you were not given. Where information is missing, '
                    'state the gap explicitly rather than assuming it.')

BASELINE_PROMPT = '''Analyze the following situation and recommend next steps.

SITUATION:
<<CTX>>
'''

BASELINE_HEDGED_PROMPT = '''Analyze the following situation and recommend next steps.
<<CLAUSE>>

SITUATION:
<<CTX>>
'''

GENERIC_PROMPT = '''Analyze the following situation using this structure:
1. Goals
2. Key facts
3. Assumptions
4. Likely causes
5. Risks
6. Recommendations
<<CLAUSE>>

SITUATION:
<<CTX>>
'''

# --- v1.0, unmodified comparison arm -------------------------------------
AXIOM_V10_PROMPT = '''Analyze the following situation by rebuilding the system from first principles.
<<CLAUSE>>

Use exactly these headings:
1. System Definition (system of interest, environment, interfaces, actors)
2. Measured Variable vs Valued Variable
3. Invariants (what must remain true regardless of the model)
4. Constraints
5. Hypotheses (candidate causal mechanisms)
6. Falsifiers (what observation would disconfirm each hypothesis)
7. Control Loop Specification (signal, comparator, actuator, delay)
8. Goodhart Check (how the metric could be won while the valued variable degrades)
9. Predictions and Perturbation Test (if X then Y within T, and what change moves the regime)
10. Abstraction and Transfer Pattern

SITUATION:
<<CTX>>
'''

# --- v0.3, discriminator correction --------------------------------------
AXIOM_V03_PROMPT = '''Analyze the following situation by rebuilding the system from first principles.
<<CLAUSE>>

APPLICABILITY TEST. First state whether this analysis should run at all. It should NOT run if the
subject is an individual person's internal state rather than a system, if the task is emotional
support, or if the required output is creative rather than diagnostic. If it should not run, say so,
explain what the appropriate response is instead, and stop.

STEP 0. EVIDENCE SUFFICIENCY. Classify the evidence as E0 (insufficient: a single aggregate figure,
no variance, no corroboration), E1 (partial: competing hypotheses formable but not separable), or
E2 (sufficient: multiple independent signals, a time series, or corroborating detail). If E0, state
what is not determinable and what data would resolve it, and stop.

If E1 or E2, work through the following. Each is a TEST, not a field to fill. Every one has a valid
negative answer, and stating the negative is a complete and correct response. Do not supply content
for a heading where the evidence does not support it. A short analysis that correctly reports
negatives is better than a long one that manufactures findings.

1. System definition and boundary. Name the measured variable(s) and the valued variable(s).
   If they are the same variable, say so explicitly and state that they are aligned.
2. Invariants, or state that none can be established from the material.
3. Constraints, or state that none are given and none can be inferred.
4. Hypotheses. IS a causal mechanism identifiable from the evidence? If not, say so and stop here.
   At E1 give candidates only, each with the evidence that would select it.
5. The cheapest observation that would discriminate between the candidates.
6. A falsifier for each hypothesis. Flag any hypothesis that cannot produce one.
7. Control loop specification (signal, comparator, actuator, delay), ONLY if a feedback loop is
   actually operating. Otherwise state that no control loop is present.
8. Metric substitution audit, in three parts.
   (a) IS THERE a route by which the metric can be won while the valued variable degrades?
       Answer yes or no first. If no, state "no metric substitution present" and go to 8(c).
   (b) If yes, what is the route, and what observation would reveal it?
   (c) Would any proposed or contemplated intervention CREATE such a route where none exists now?
9. Predictions. State only as many "if X, then Y within T" predictions as the evidence supports.
   There is no required number. Zero is a valid answer.
10. Transferable pattern, ONLY if one genuinely generalises beyond this case, stated with its
    falsifier and a domain that would test it. Otherwise state that this case is particular.

SITUATION:
<<CTX>>
'''

CONDITIONS = {
    'baseline': (BASELINE_PROMPT, False),
    'baseline_hedged': (BASELINE_HEDGED_PROMPT, True),
    'generic_structured': (GENERIC_PROMPT, True),
    'axiom_v10': (AXIOM_V10_PROMPT, True),
    'axiom_v03': (AXIOM_V03_PROMPT, True),
}
assert sorted(CONDITIONS) == sorted(PREREGISTRATION['conditions'])

def build_prompt(cond, ctx):
    template, hedged = CONDITIONS[cond]
    out = template.replace('<<CLAUSE>>', EPISTEMIC_CLAUSE if hedged else '')
    return out.replace('<<CTX>>', ctx)

In [8]:
# cue: 'none'       = no condition is prompted for this
#      'both'       = generic and both axiom arms are prompted for it
#      'axiom_only' = mirrors an AXIOM heading, excluded from primary_total
RUBRIC = [
  ('cause_match',             'both',       'the proposed cause matches the actual mechanism in the key; for cases where the key says the cause is not determinable, only an explicit statement of indeterminacy scores 2'),
  ('no_invention',            'none',       'asserts no specific facts that were not present in the material the analyst received'),
  ('alternative_explanations','none',       'raises and weighs at least one credible competing explanation'),
  ('calibration',             'none',       'confidence is proportionate to the evidence available; says what cannot be determined; does not over-structure thin or irreducibly uncertain evidence'),
  ('null_correctness',        'none',       'where the reference analysis states that a cause is not determinable, that no metric-outcome gap exists, that no control loop is operating, or that the case does not generalise, the output says so explicitly; asserting content where the key says none exists scores 0; brevity is not penalised'),
  ('stakeholder_impact',      'none',       'identifies who bears the harm and how'),
  ('intervention_fit',        'both',       'recommended actions act on the cause it identified, and are feasible given stated limits'),
  ('prioritisation',          'none',       'orders or triages actions by leverage or feasibility rather than listing them flat'),
  ('internal_consistency',    'none',       'conclusions follow from its own stated facts, with no self-contradiction'),
  ('metric_outcome_gap',      'axiom_only', 'separates the number being tracked from the outcome that matters, and does not assert such a gap where none exists'),
  ('stable_condition',        'axiom_only', 'identifies a case-specific fact that holds regardless of what records say'),
  ('operating_limits',        'axiom_only', 'identifies the relevant operational limits'),
  ('feedback_path',           'axiom_only', 'traces what is measured, what it is compared against, who changes behaviour, and the delay'),
  ('disconfirming_test',      'axiom_only', 'proposes an observation that could actually show its explanation is wrong'),
]

DIMENSIONS = [d for d, _c, _x in RUBRIC]
PRIMARY_DIMS = [d for d, c, _x in RUBRIC if c in ('none', 'both')]
CUED_DIMS    = [d for d, c, _x in RUBRIC if c == 'axiom_only']

assert len(PRIMARY_DIMS) == 9 and len(CUED_DIMS) == 5
print('primary_total max', 2 * len(PRIMARY_DIMS), '| cued_total max', 2 * len(CUED_DIMS))

primary_total max 18 | cued_total max 10


In [ ]:
import itertools, json, uuid, random, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

if not EXTERNAL_WITNESS:
    raise SystemExit('Set EXTERNAL_WITNESS in Cell 1 and commit the design before generating.')
if 'REPLACE_BEFORE_HASHING' in json.dumps(PREREGISTRATION):
    raise SystemExit('Provider placeholder still in the pre-registration. Verify and re-hash.')

CASE_BY_ID = {c['id']: c for c in CASES}

# ---------------------------------------------------------------- preflight
STRATEGIES = [
    ('qwen_kwargs', {'extra_body': {'chat_template_kwargs': {'enable_thinking': False}}}),
    ('glm_body',    {'extra_body': {'thinking': {'type': 'disabled'}}}),
    ('none',        {}),
]

PREFLIGHT_PROMPT = ('A team hit its target every quarter for two years, then missed by 40 percent. '
                    'Give a two sentence assessment.')

def preflight(model):
    '''Pin to the hashed provider. NO router fallback: in v0.2 the judge silently ran off-hash.'''
    prov = PROVIDER_MAP.get(model)
    PROVIDER_FOR[model] = prov
    fallback = None
    for name, kw in STRATEGIES:
        try:
            r = get_client(model).chat_completion(
                messages=[{'role': 'user', 'content': PREFLIGHT_PROMPT}],
                model=model, max_tokens=600, temperature=0.0, **kw)
            raw = r.choices[0].message.content or ''
            thinks = bool(THINK_RE.search(raw)) or '<think>' in raw
            print(model, '| provider:', prov, '| strategy:', name,
                  '| think emitted:', thinks, '| finish:', r.choices[0].finish_reason)
            if not thinks:
                NO_THINK[model] = kw
                print('  -> selected', name, 'on', prov)
                return True
            if fallback is None:
                fallback = kw
        except Exception as e:
            print(model, '| provider:', prov, '| strategy', name,
                  'failed:', error_detail(e)[:400])
    if fallback is not None:
        NO_THINK[model] = fallback
        print(model, '| WARNING: thinking could not be suppressed on the pinned provider. '
                     'strip_think() removes it from the text but the tokens were generated and '
                     'completion_tokens is inflated. Stop and investigate.')
        return True
    return False

for m in {GEN_MODEL, JUDGE_MODEL, AUDIT_MODEL}:
    if not preflight(m):
        raise SystemExit('Preflight failed for ' + m + ' on pinned provider '
                         + str(PROVIDER_MAP.get(m)) + '. Do NOT fall back to the router: that '
                         'would put the run off-hash, which is the v0.2 deviation. Correct the '
                         'provider in Cell 1 and re-hash under a new witness.')

# ---------------------------------------------------------------- grid
random.seed(SEED_BASE)
CONTEXT_LEVELS = list(PREREGISTRATION['context_levels'])     # ['full', 'fragmented']
grid = list(itertools.product([c['id'] for c in CASES], CONDITIONS.keys(),
                              CONTEXT_LEVELS, range(RUNS_PER_CELL)))
random.shuffle(grid)                      # decorrelates seed from condition

specs = [{'i': i, 'case_id': cid, 'condition': cond, 'context_level': lvl, 'rep': rep,
          'seed': SEED_BASE + i, 'cell_key': cid + '|' + cond + '|' + lvl + '|' + str(rep)}
         for i, (cid, cond, lvl, rep) in enumerate(grid)]
print('\ncells to run:', len(specs))

# ---------------------------------------------------------------- resume
out_path = os.path.join(OUTPUT_DIR, 'generations.jsonl')
done = {}
if os.path.exists(out_path):
    with open(out_path) as fh:
        for ln in fh:
            try:
                r = json.loads(ln)
                if r.get('status') == 'ok' and r.get('prereg_sha256') == PREREG_HASH:
                    done[r['cell_key']] = r
            except Exception:
                pass
    print('resuming, already complete under this hash:', len(done))
    print('Rows written under a different hash are ignored automatically.')

todo = [s for s in specs if s['cell_key'] not in done]

lock = threading.Lock()
err_count = {'n': 0}
ABORT = threading.Event()

def run_one(s):
    if ABORT.is_set():
        return None
    case = CASE_BY_ID[s['case_id']]
    ctx = get_context(case, s['context_level'])
    prompt = build_prompt(s['condition'], ctx)
    try:
        text, usage, finish = generate(GEN_MODEL, SYSTEM_NEUTRAL, prompt, seed=s['seed'])
        truncated = (finish == 'length')
        status = 'truncated' if truncated else ('empty' if not text.strip() else 'ok')
        with lock:
            err_count['n'] = 0
    except Exception as e:
        text, usage, finish, truncated, status = '', {}, None, False, 'error: ' + str(e)
        with lock:
            err_count['n'] += 1
            print('ERROR', err_count['n'], ':', str(e)[:400])
            if err_count['n'] >= 5:
                ABORT.set()
                print('CIRCUIT BREAKER TRIPPED after 5 consecutive errors')
    return {
        'output_id': uuid.uuid4().hex[:12],
        'prereg_sha256': PREREG_HASH,
        'cell_key': s['cell_key'],
        'case_id': s['case_id'],
        'kind': case['kind'],
        'adversarial': bool(case.get('adversarial')),
        'false_positive_test': bool(case.get('false_positive_test')),
        'condition': s['condition'],
        'context_level': s['context_level'],
        'rep': s['rep'],
        'seed': s['seed'],
        'provider': PROVIDER_FOR.get(GEN_MODEL),
        'finish_reason': finish,
        'truncated': truncated,
        'status': status,
        'context_given': ctx,
        'output': text,
        'completion_tokens': (usage or {}).get('completion_tokens'),
        'prompt_tokens': (usage or {}).get('prompt_tokens'),
    }

new = []
with open(out_path, 'a') as fh, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = [ex.submit(run_one, s) for s in todo]
    for n, fut in enumerate(as_completed(futs), 1):
        rec = fut.result()
        if rec is None:
            continue
        new.append(rec)
        with lock:
            fh.write(json.dumps(rec) + '\n')
            fh.flush()
        if n % 20 == 0:
            print(n, '/', len(todo))

print('completed this pass:', len(new), '| skipped after abort:', len(todo) - len(new))
if ABORT.is_set():
    raise SystemExit('Aborted on 5 consecutive generation errors. See the ERROR lines above. '
                     'Rerun this cell to resume from where it stopped.')

runs = list(done.values()) + new
runs.sort(key=lambda r: r['cell_key'])
df = pd.DataFrame(runs)

print('\n=== truncation rate by condition (asymmetry here is a confound) ===')
trunc_by_cond = df.groupby('condition').truncated.mean()
print(trunc_by_cond.round(3))

_cap = PREREGISTRATION['max_acceptable_truncation_rate']
if len(trunc_by_cond) and trunc_by_cond.max() > _cap >= trunc_by_cond.min():
    raise SystemExit('VOID: truncation is asymmetric across conditions (max '
                     + str(round(trunc_by_cond.max(), 3)) + ' vs min '
                     + str(round(trunc_by_cond.min(), 3)) + ', cap ' + str(_cap) + '). '
                     'Recalibrate max_new_tokens under a new pre-registration hash.')

print('\nexcluded:', (df.status != 'ok').sum(), 'of', len(df))
print(df[df.status == 'ok'].groupby(['condition', 'context_level']).size())

In [ ]:
import re

SECTION_WORDS = (r'system definition[^:\n]*|boundary|measured variable[^:\n]*|valued variable[^:\n]*|'
                 r'invariants?|constraints?|hypothes\w+|falsifiers?|control loop[^:\n]*|'
                 r'goodhart[^:\n]*|metric substitution[^:\n]*|applicability[^:\n]*|'
                 r'evidence sufficiency[^:\n]*|step 0[^:\n]*|transferable pattern[^:\n]*|'
                 r'predictions?[^:\n]*|perturbation[^:\n]*|abstraction[^:\n]*|transfer pattern|'
                 r'goals?|key facts?|assumptions?|likely causes?|causes?|risks?|recommendations?|'
                 r'next steps?|analysis|summary|conclusion')

LINE_HEADINGS = [
    re.compile(r'(?i)^\s*#{1,6}\s*\d*[\.\)]?\s*(?:' + SECTION_WORDS + r')\s*:?\s*$'),
    re.compile(r'(?i)^\s*[-*]?\s*\d+[\.\)]\s*(?:' + SECTION_WORDS + r')\s*:?\s*$'),
    re.compile(r'(?i)^\s*[-*]?\s*\*{0,2}\s*\d*[\.\)]?\s*(?:' + SECTION_WORDS + r')\s*\*{0,2}\s*:?\s*$'),
]

INLINE_HEADING = re.compile(
    r'(?i)^\s*(?:[-*]\s*)?(?:#{1,6}\s*)?\*{0,2}\s*\d*[\.\)]?\s*(?:' + SECTION_WORDS + r')\s*\*{0,2}\s*:\s*')
LEADING_ENUM = re.compile(r'^\s*(?:\d+[\.\)]|[-*+•]|\([a-c]\))\s*')
EMPHASIS = re.compile(r'\*{1,3}|_{1,3}|`+')
FRAMEWORK_WORDS = re.compile(r'(?i)\b(axiom|first principles|the framework|v0\.3|v1\.0)\b')
# E0/E1/E2 labels appear only in the v0.3 arm and would give it away outright.
EVIDENCE_LABELS = re.compile(r'(?i)\bE[012]\b(?:\s*\([^)]*\))?')

def deidentify(text):
    '''Remove all condition-revealing scaffolding while preserving substantive content.'''
    kept = []
    for ln in text.split('\n'):
        if any(p.match(ln) for p in LINE_HEADINGS):
            continue
        ln = INLINE_HEADING.sub('', ln)
        ln = LEADING_ENUM.sub('', ln)
        ln = EMPHASIS.sub('', ln)
        ln = FRAMEWORK_WORDS.sub('the approach', ln)
        ln = EVIDENCE_LABELS.sub('the evidence level', ln)
        ln = ln.strip()
        if ln:
            kept.append('- ' + ln)
    return '\n'.join(kept).strip()

In [ ]:
import numpy as np

AUDIT_SYSTEM = ('You classify text by which instruction style produced it. Return only a single '
                'label from the list given. No explanation.')

# D and E are described symmetrically and neither description hints at which is the revision.
AUDIT_TEMPLATE = '''Five instruction styles were used to produce analytical outputs:

A = unstructured: "analyze and recommend next steps"
B = unstructured plus an instruction to flag missing information
C = a six-heading structure: goals, key facts, assumptions, causes, risks, recommendations
D = a ten-part systems structure covering variables, invariants, loops and metric checks
E = a ten-part systems structure covering variables, invariants, loops and metric checks

D and E differ only in wording, not in subject matter.

Below is one output with formatting removed. Which style produced it?
Answer with exactly one letter: A, B, C, D or E.

OUTPUT:
<<OUT>>
'''

LABEL_TO_COND = {'A': 'baseline', 'B': 'baseline_hedged', 'C': 'generic_structured',
                 'D': 'axiom_v10', 'E': 'axiom_v03'}

audit_pool = df[df.status == 'ok'].sample(min(100, (df.status == 'ok').sum()),
                                          random_state=SEED_BASE)
hits, total, guesses = 0, 0, []
for rec in audit_pool.to_dict('records'):
    p = AUDIT_TEMPLATE.replace('<<OUT>>', deidentify(rec['output']))
    try:
        txt, _u, _f = generate(AUDIT_MODEL, AUDIT_SYSTEM, p,
                               max_new_tokens=AUDIT_MAX_TOKENS, temperature=0.0)
    except Exception:
        continue
    m = re.search(r'\b([ABCDE])\b', txt.strip().upper())
    if not m:
        continue
    guess = LABEL_TO_COND[m.group(1)]
    guesses.append({'true': rec['condition'], 'guess': guess})
    hits += int(guess == rec['condition'])
    total += 1

gdf = pd.DataFrame(guesses)
BLIND_AUDIT_ACC = hits / total if total else float('nan')

if len(gdf):
    recall = gdf.groupby('true').apply(lambda g: (g.guess == g.name).mean())
    marginal = gdf.guess.value_counts(normalize=True)
    print('=== per-condition recall (the gate) ===')
    print(recall.round(3))
    print('\n=== auditor guess marginals (a degenerate guesser inflates one recall) ===')
    print(marginal.round(3))
    print('\npooled accuracy (reported, NOT the gate):', round(BLIND_AUDIT_ACC, 3))
    MAX_RECALL = float(recall.max())
    blinding_ok = bool(MAX_RECALL <= PREREGISTRATION['max_blinding_condition_recall'])
    print('\nmax per-condition recall:', round(MAX_RECALL, 3),
          '| threshold:', PREREGISTRATION['max_blinding_condition_recall'])
else:
    MAX_RECALL, blinding_ok = float('nan'), False
    print('no audit data')

print('blinding_ok:', blinding_ok)
if not blinding_ok:
    print('At least one condition is recoverable from deidentified text. Strengthen deidentify() '
          'and rerun before treating any score as blind.')
if len(gdf):
    print('\nconfusion:')
    print(gdf.groupby('true').guess.value_counts())

In [ ]:
JUDGE_SYSTEM = ('You are a strict evaluator. Score analytical outputs on content only. Never award '
                'points for headings, terminology, framework vocabulary, structure or length. A '
                'correct point made in plain prose scores the same as one made under a heading. '
                'A short answer that correctly reports that something is not determinable or not '
                'present is worth as much as a long one that finds something. Naming a concept '
                'without correctly reconstructing it scores 0 or 1. Return JSON only.')

JUDGE_TEMPLATE = '''MATERIAL THE ANALYST RECEIVED (this is all they were given):
<<CTX>>

REFERENCE ANALYSIS (ground truth, withheld from the analyst):
<<KEY>>

ANALYST OUTPUT:
<<OUT>>

Score each dimension 0, 1 or 2.
0 = missing or wrong, 1 = partial or generic, 2 = correct and specific to this situation.
Where the reference analysis says something is not determinable, not present or not applicable, an
output that correctly says so scores 2 and an output that confidently asserts a mechanism scores 0.
Do not reward length. Do not penalise an output for being short if it is correct.

Dimensions:
<<DIMS>>

Also return:
- unsupported_claims: integer count of specific factual claims in the output that are absent from
  the material the analyst received and not supported by the reference analysis.
- restraint: 0, 1 or 2. 2 means the output stayed within what the evidence supports and did not
  impose structure, mechanisms or causal models the situation does not warrant. 0 means it
  over-structured or over-claimed.
- false_metric_failure: 1 if the output asserts metric gaming, gaming of an incentive, or a
  decoupling between a measure and the real outcome that the reference analysis says is NOT
  present, otherwise 0. A forward-looking warning that a PROPOSED intervention would create such
  a problem is NOT a false metric failure; score 0 for that.
- null_stated_explicitly: 1 if the output explicitly states the relevant negative (that a cause is
  not determinable, that the measure and the outcome are aligned, that no control loop is
  operating, or that the case does not generalise) where the reference analysis says so.
  0 if it merely omits the topic without stating the negative, or if no negative applies.

Return a single JSON object with exactly these keys and integer values:
<<KEYS>>
'''

DIM_TEXT = '\n'.join('- ' + d + ': ' + desc for d, _c, desc in RUBRIC)
KEYS_TEXT = json.dumps({**{d: 0 for d in DIMENSIONS},
                        'unsupported_claims': 0, 'restraint': 0,
                        'false_metric_failure': 0, 'null_stated_explicitly': 0})

NEG_JUDGE_TEMPLATE = '''MATERIAL THE ANALYST RECEIVED:
<<CTX>>

CORRECT ANSWER:
total = <<TOTAL>>
explanation = <<EXPL>>

ANALYST OUTPUT:
<<OUT>>

Score 0, 1 or 2 on each:
- answer_correct: is the stated quarter total correct
- explanation_correct: does it identify the stated fact that explains the May figure
- conciseness_fit: is the response proportionate to a simple extraction question

Return a single JSON object with exactly these keys and integer values:
{"answer_correct":0,"explanation_correct":0,"conciseness_fit":0}
'''

NEG_DIMENSIONS = ['answer_correct', 'explanation_correct', 'conciseness_fit']

def parse_json_block(text):
    m = re.search(r'\{.*\}', text, re.S)
    if not m:
        raise ValueError('no json found')
    return json.loads(m.group(0))

def judge_main(case, ctx, output_text, retries=2):
    key_prose = '\n\n'.join(k.replace('_', ' ') + ': ' + v for k, v in case['key'].items())
    p = (JUDGE_TEMPLATE.replace('<<CTX>>', ctx)
                       .replace('<<KEY>>', key_prose)
                       .replace('<<OUT>>', deidentify(output_text))
                       .replace('<<DIMS>>', DIM_TEXT)
                       .replace('<<KEYS>>', KEYS_TEXT))
    for _ in range(retries + 1):
        txt, _u, _f = generate(JUDGE_MODEL, JUDGE_SYSTEM, p,
                               max_new_tokens=JUDGE_MAX_TOKENS, temperature=0.0)
        try:
            return parse_json_block(txt)
        except Exception:
            continue
    return None

def judge_neg(case, ctx, output_text, retries=2):
    p = (NEG_JUDGE_TEMPLATE.replace('<<CTX>>', ctx)
                           .replace('<<TOTAL>>', case['key']['correct_total'])
                           .replace('<<EXPL>>', case['key']['correct_explanation'])
                           .replace('<<OUT>>', deidentify(output_text)))
    for _ in range(retries + 1):
        txt, _u, _f = generate(JUDGE_MODEL, JUDGE_SYSTEM, p,
                               max_new_tokens=NEG_JUDGE_MAX_TOKENS, temperature=0.0)
        try:
            return parse_json_block(txt)
        except Exception:
            continue
    return None

In [ ]:
SCORE_CKPT = os.path.join(OUTPUT_DIR, 'scores_checkpoint.jsonl')

score_records, neg_records, judge_failures = [], [], []
scored_ids = set()
if os.path.exists(SCORE_CKPT):
    with open(SCORE_CKPT) as fh:
        for ln in fh:
            try:
                r = json.loads(ln)
            except Exception:
                continue
            if r.get('prereg_sha256') != PREREG_HASH:
                continue
            scored_ids.add(r['output_id'])
            (neg_records if r.get('kind') == 'negative_control' else score_records).append(r)
    print('resuming scoring, already done under this hash:', len(scored_ids))

def _ckpt(row):
    with open(SCORE_CKPT, 'a') as fh:
        fh.write(json.dumps(row) + '\n')

order = df[df.status == 'ok'].sample(frac=1.0, random_state=SEED_BASE).to_dict('records')
if not order:
    raise RuntimeError('Nothing to score: no generations with status ok. Check Cell 8 output.')

order = [r for r in order if r['output_id'] not in scored_ids]
print('to score this pass:', len(order))

for i, rec in enumerate(order):
    case = CASE_BY_ID[rec['case_id']]
    meta = {k: rec[k] for k in ('output_id', 'case_id', 'kind', 'adversarial',
                               'false_positive_test', 'condition', 'context_level',
                               'rep', 'completion_tokens')}
    meta['prereg_sha256'] = PREREG_HASH
    if case['kind'] == 'negative_control':
        s = judge_neg(case, rec['context_given'], rec['output'])
        if s is None:
            judge_failures.append({'output_id': rec['output_id'], 'case_id': rec['case_id'],
                                   'condition': rec['condition'],
                                   'context_level': rec['context_level'], 'kind': case['kind']})
            continue
        row = dict(meta)
        row.update({d: int(s.get(d, 0)) for d in NEG_DIMENSIONS})
        row['neg_total'] = sum(row[d] for d in NEG_DIMENSIONS)
        neg_records.append(row)
        _ckpt(row)
    else:
        s = judge_main(case, rec['context_given'], rec['output'])
        if s is None:
            judge_failures.append({'output_id': rec['output_id'], 'case_id': rec['case_id'],
                                   'condition': rec['condition'],
                                   'context_level': rec['context_level'], 'kind': case['kind']})
            continue
        row = dict(meta)
        row.update({d: int(s.get(d, 0)) for d in DIMENSIONS})
        row['unsupported_claims'] = int(s.get('unsupported_claims', 0))
        row['restraint'] = int(s.get('restraint', 0))
        row['false_metric_failure'] = int(s.get('false_metric_failure', 0))
        row['null_stated_explicitly'] = int(s.get('null_stated_explicitly', 0))
        row['primary_total'] = sum(row[d] for d in PRIMARY_DIMS)
        row['cued_total'] = sum(row[d] for d in CUED_DIMS)
        score_records.append(row)
        _ckpt(row)
    if i % 20 == 0:
        print('scored', i, '/', len(order))

if not score_records:
    raise RuntimeError('Every main-case judge call failed to parse. Inspect judge_failures.csv '
                       'and check finish_reason on a raw judge call before rerunning.')

scores_df = pd.DataFrame(score_records)
neg_df = pd.DataFrame(neg_records)
fail_df = pd.DataFrame(judge_failures)
scores_df.to_csv(os.path.join(OUTPUT_DIR, 'scores_main.csv'), index=False)
neg_df.to_csv(os.path.join(OUTPUT_DIR, 'scores_negative_control.csv'), index=False)
fail_df.to_csv(os.path.join(OUTPUT_DIR, 'judge_failures.csv'), index=False)

print(scores_df.shape, neg_df.shape)
print('judge failures:', len(fail_df), 'of', len(order))
if len(fail_df):
    print(fail_df.groupby('condition').size())
    print('Uneven parse failure across conditions is a confound, not noise.')

In [ ]:
from sklearn.metrics import cohen_kappa_score

HUMAN_SCORER_ID = ''          # required, and must not be the framework author
HUMAN_SHEET = os.path.join(OUTPUT_DIR, 'human_blind_scoring_sheet.csv')

# --- export, stratified so null cases are represented ---
null_pool = scores_df[scores_df.false_positive_test]
other_pool = scores_df[~scores_df.false_positive_test]
sample_ids = (null_pool.sample(min(30, len(null_pool)), random_state=7)['output_id'].tolist()
              + other_pool.sample(min(30, len(other_pool)), random_state=7)['output_id'].tolist())
blind = df[df.output_id.isin(sample_ids)].copy()
blind['blinded_output'] = blind['output'].apply(deidentify)
blind = blind[['output_id', 'context_given', 'blinded_output']].sample(frac=1.0, random_state=11)
for d in DIMENSIONS:
    blind[d] = ''
blind['false_metric_failure'] = ''
blind.to_csv(HUMAN_SHEET, index=False)
print('wrote', HUMAN_SHEET, 'with', len(blind), 'rows. Have an independent scorer fill it in.')

# --- agreement, run after the sheet comes back filled ---
def compute_irr(path=HUMAN_SHEET):
    h = pd.read_csv(path)
    h = h.dropna(subset=DIMENSIONS, how='all')
    if h.empty or not HUMAN_SCORER_ID:
        return float('nan'), False, None
    m = h.merge(scores_df, on='output_id', suffixes=('_h', '_j'))
    rows = []
    for d in DIMENSIONS:
        a = pd.to_numeric(m[d + '_h'], errors='coerce')
        b = pd.to_numeric(m[d + '_j'], errors='coerce')
        ok = a.notna() & b.notna()
        if ok.sum() < 5 or a[ok].nunique() < 2 or b[ok].nunique() < 2:
            rows.append({'dimension': d, 'weighted_kappa': float('nan'), 'n': int(ok.sum())})
            continue
        k = cohen_kappa_score(a[ok].astype(int), b[ok].astype(int), weights='quadratic')
        rows.append({'dimension': d, 'weighted_kappa': k, 'n': int(ok.sum())})
    tab = pd.DataFrame(rows)
    mean_k = tab.weighted_kappa.mean(skipna=True)
    return mean_k, mean_k >= PREREGISTRATION['min_mean_weighted_kappa'], tab

MEAN_KAPPA, irr_ok, irr_table = compute_irr()
print('\nmean quadratic weighted kappa (human vs judge):', MEAN_KAPPA)
print('irr_ok:', irr_ok, '| threshold:', PREREGISTRATION['min_mean_weighted_kappa'])
if irr_table is not None:
    print(irr_table.round(3).to_string(index=False))
if not irr_ok:
    print('Judge scores are not corroborated. No verdict may be issued on judge scores alone.')

In [ ]:
LEVEL_ORDER = {lvl: i for i, lvl in enumerate(PREREGISTRATION['context_levels'])}
LEVELS = list(PREREGISTRATION['context_levels'])
main = scores_df.copy()
main['level_idx'] = main.context_level.map(LEVEL_ORDER)

print('=== PRIMARY OUTCOME: false metric-failure rate on null cases (lower is better) ===')
nulls = main[main.false_positive_test]
fp_rate = nulls.groupby('condition').false_metric_failure.mean()
print(fp_rate.round(3))
print('\nby case:')
print(nulls.pivot_table(index='case_id', columns='condition',
                        values='false_metric_failure', aggfunc='mean').round(3))

false_goodhart_ok_v03 = bool(fp_rate.get('axiom_v03', 1.0)
                             <= PREREGISTRATION['max_false_goodhart_rate_axiom'])
print('\naxiom_v03 within the 0.30 threshold:', false_goodhart_ok_v03)
print('axiom_v10 within the 0.30 threshold:',
      bool(fp_rate.get('axiom_v10', 1.0) <= PREREGISTRATION['max_false_goodhart_rate_axiom']))

print('\n=== explicit null statements on null cases (higher is better) ===')
print(nulls.groupby('condition').null_stated_explicitly.mean().round(3))

print('\n=== SENSITIVITY GUARD: cause_match on true-positive cases C1-C5 ===')
tp = main[main.case_id.isin(TP_IDS)]
cm = tp.groupby('condition').cause_match.mean()
print(cm.round(3))
_v10 = max(float(cm.get('axiom_v10', 0.0)), 1e-9)
retention = float(cm.get('axiom_v03', 0.0)) / _v10
sensitivity_ok = bool(retention >= PREREGISTRATION['min_true_positive_retention'])
print('retention (v03 / v10):', round(retention, 3),
      '| threshold:', PREREGISTRATION['min_true_positive_retention'],
      '| sensitivity_ok:', sensitivity_ok)
if not sensitivity_ok:
    print('The revision reduced false positives by losing sensitivity. That is refusal, '
          'not discrimination.')

print('\n=== adversarial restraint by condition (max 2) ===')
adv = main[main.adversarial]
adv_restraint = adv.groupby('condition').restraint.mean()
print(adv_restraint.round(2))
print('\nrestraint by case:')
print(adv.pivot_table(index='case_id', columns='condition',
                      values='restraint', aggfunc='mean').round(2))
adversarial_ok = bool(adv_restraint.get('axiom_v03', 0)
                      >= PREREGISTRATION['min_adversarial_restraint_mean'])
print('\nadversarial_ok (reported as an OUTCOME, not a gate):', adversarial_ok)

print('\n=== primary_total (condition-neutral, max 18) by condition x context level ===')
pv = main.pivot_table(index='condition', columns='context_level',
                      values='primary_total', aggfunc='mean')[LEVELS]
print(pv.round(2))
print('\n=== degradation, full minus fragmented (SECONDARY: confounded by floor effects) ===')
print((pv['full'] - pv['fragmented']).sort_values().round(2))

print('\n=== cued_total (mirrors AXIOM headings, DIAGNOSTIC ONLY, cannot support H1) ===')
print(main.pivot_table(index='condition', columns='context_level',
                       values='cued_total', aggfunc='mean')[LEVELS].round(2))

print('\n=== unsupported claims and calibration ===')
print(main.groupby('condition').unsupported_claims.mean().round(2))
print(main.groupby('condition').calibration.mean().round(2))

if len(neg_df):
    ncmp = neg_df.groupby('condition').neg_total.mean()
    print('\n=== negative control, own rubric (max 6). AXIOM advantage here implies rubric bias ===')
    print(ncmp.round(2))
    negative_control_ok = bool(max(ncmp.get('axiom_v03', 0), ncmp.get('axiom_v10', 0))
                               - ncmp.get('baseline', 0)
                               <= PREREGISTRATION['max_negative_control_advantage_points'])
else:
    negative_control_ok = False
print('negative_control_ok:', negative_control_ok)

from scipy import stats
tok = main.dropna(subset=['completion_tokens']).copy()
if len(tok):
    print('\n=== mean completion tokens by condition ===')
    print(tok.groupby('condition').completion_tokens.mean().round(0))
    print('If v0.3 is working, it should be SHORTER than v1.0 on null cases:')
    print(tok[tok.false_positive_test].groupby('condition').completion_tokens.mean().round(0))
    tokens_available = True
else:
    tokens_available = False
    print('\nno token usage returned')

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# ---------------- PRIMARY: specificity, axiom_v03 vs axiom_v10 on null cases
pair = nulls[nulls.condition.isin(['axiom_v03', 'axiom_v10'])]
tab = pd.crosstab(pair.condition, pair.false_metric_failure)
for c in (0, 1):
    if c not in tab.columns:
        tab[c] = 0
tab = tab[[0, 1]]
print('=== 2x2: condition by false_metric_failure (null cases only) ===')
print(tab)

_odds, p_fisher = stats.fisher_exact(
    [[int(tab.loc['axiom_v03', 1]), int(tab.loc['axiom_v03', 0])],
     [int(tab.loc['axiom_v10', 1]), int(tab.loc['axiom_v10', 0])]],
    alternative='less')
print('\nFisher exact, one-sided (v03 fewer false positives than v10): p =', round(p_fisher, 5))
print('NOTE: observations cluster within case, so this p is anticonservative. '
      'The case-level test below is the honest one.')

# robustness: case-level paired, n = number of null cases
case_fp = nulls.groupby(['case_id', 'condition']).false_metric_failure.mean().unstack()
print('\n=== false-positive rate by null case and condition ===')
print(case_fp.round(3))
if {'axiom_v03', 'axiom_v10'}.issubset(case_fp.columns):
    cp = case_fp[['axiom_v03', 'axiom_v10']].dropna()
    try:
        _s, p_case = stats.wilcoxon(cp['axiom_v03'], cp['axiom_v10'], alternative='less')
    except ValueError:
        p_case = float('nan')
    print('\ncase-level paired Wilcoxon, n =', len(cp), '| p =', p_case)
    print('With', len(cp), 'null cases this test cannot reach p < 0.05 by itself. It is a '
          'direction check, not a significance test. Report both.')

# ---------------- SECONDARY: all conditions against axiom_v10 on specificity
rows = []
for cond in ['baseline', 'baseline_hedged', 'generic_structured', 'axiom_v03']:
    sub = nulls[nulls.condition.isin([cond, 'axiom_v10'])]
    t = pd.crosstab(sub.condition, sub.false_metric_failure)
    for c in (0, 1):
        if c not in t.columns:
            t[c] = 0
    _o, p = stats.fisher_exact([[int(t.loc[cond, 1]), int(t.loc[cond, 0])],
                                [int(t.loc['axiom_v10', 1]), int(t.loc['axiom_v10', 0])]],
                               alternative='less')
    rows.append({'contrast': cond + ' vs axiom_v10', 'p_raw': p})
sec = pd.DataFrame(rows)
sec['p_holm'] = multipletests(sec.p_raw, method='holm')[1]
print('\n=== secondary contrasts on specificity, Holm corrected ===')
print(sec.round(4).to_string(index=False))

# ---------------- SECONDARY: context degradation, mixed effects
main['level_idx'] = main.context_level.map(LEVEL_ORDER)
model_df = main.copy()
model_df['condition'] = pd.Categorical(
    model_df.condition,
    categories=['axiom_v03', 'axiom_v10', 'baseline', 'baseline_hedged', 'generic_structured'])
try:
    md = smf.mixedlm('primary_total ~ C(condition) * level_idx', model_df,
                     groups=model_df['case_id'])
    mf = md.fit()
    print('\n=== mixed effects, reference = axiom_v03 (SECONDARY) ===')
    print(mf.summary())
    print('A positive condition coefficient means that condition scored HIGHER than axiom_v03.')
except Exception as e:
    print('\nmixedlm failed:', e)

# ---------------- token control on the primary outcome
if tokens_available:
    tk = nulls.dropna(subset=['completion_tokens']).copy()
    tk['log_tokens'] = np.log(tk.completion_tokens.clip(lower=1))
    try:
        lg = smf.logit('false_metric_failure ~ log_tokens', data=tk).fit(disp=0)
        print('\n=== does output length alone predict a false positive? ===')
        print(lg.summary().tables[1])
        print('If log_tokens is significant, length is a confound and the specificity result '
              'must be reported adjusted for it.')
    except Exception as e:
        print('\nlogit failed:', e)

primary_supported = bool(p_fisher < PREREGISTRATION['alpha'] and false_goodhart_ok_v03)
print('\nprimary_supported:', primary_supported)
print('Requires BOTH a significant reduction against v1.0 AND an absolute rate at or below 0.30. '
      'A significant improvement from 1.00 to 0.60 is still a broken detector.')

In [ ]:
import matplotlib.pyplot as plt

COND_ORDER = ['baseline', 'baseline_hedged', 'generic_structured', 'axiom_v10', 'axiom_v03']
avail = [c for c in COND_ORDER if c in set(main.condition)]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. PRIMARY: false positive rate on null cases
ax = axes[0][0]
vals = [fp_rate.get(c, np.nan) for c in avail]
cols = ['#c0392b' if (v is not None and v > PREREGISTRATION['max_false_goodhart_rate_axiom'])
        else '#27ae60' for v in vals]
ax.bar(range(len(avail)), vals, color=cols)
ax.axhline(PREREGISTRATION['max_false_goodhart_rate_axiom'], ls='--', c='k', lw=1,
           label='pre-registered threshold 0.30')
ax.set_xticks(range(len(avail)))
ax.set_xticklabels(avail, rotation=20, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('false metric-failure rate')
ax.set_title('PRIMARY: false positives on null cases (lower is better)')
ax.legend(fontsize=8)

# 2. sensitivity guard
ax = axes[0][1]
ax.bar(range(len(avail)), [cm.get(c, np.nan) for c in avail], color='#2980b9')
ax.set_xticks(range(len(avail)))
ax.set_xticklabels(avail, rotation=20, ha='right')
ax.set_ylim(0, 2)
ax.set_ylabel('mean cause_match (max 2)')
ax.set_title('SENSITIVITY GUARD: true-positive cases C1-C5 (higher is better)')

# 3. specificity vs sensitivity, the actual trade-off
ax = axes[1][0]
for c in avail:
    ax.scatter(fp_rate.get(c, np.nan), cm.get(c, np.nan), s=90)
    ax.annotate(c, (fp_rate.get(c, np.nan), cm.get(c, np.nan)),
                fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.axvline(PREREGISTRATION['max_false_goodhart_rate_axiom'], ls='--', c='k', lw=1)
ax.set_xlabel('false positive rate on null cases (want low)')
ax.set_ylabel('cause_match on true positives (want high)')
ax.set_title('The trade-off. Top left is a working discriminator.')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0, 2)

# 4. per-null-case detail
ax = axes[1][1]
case_fp_plot = nulls.pivot_table(index='case_id', columns='condition',
                                 values='false_metric_failure', aggfunc='mean')
case_fp_plot = case_fp_plot[[c for c in avail if c in case_fp_plot.columns]]
case_fp_plot.plot(kind='bar', ax=ax, width=0.8)
ax.axhline(PREREGISTRATION['max_false_goodhart_rate_axiom'], ls='--', c='k', lw=1)
ax.set_ylabel('false metric-failure rate')
ax.set_title('Per null case')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=7)
ax.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'results_v03.png'), dpi=150)
plt.show()

In [ ]:
gates = {
    'blinding_ok': blinding_ok,
    'irr_ok': irr_ok,
    'negative_control_ok': negative_control_ok,
    'sensitivity_ok': sensitivity_ok,
}
print('=== VALIDITY GATES ===')
for k, v in gates.items():
    print(' ', k, ':', v)

failed = [k for k, v in gates.items() if not v]

print('\n=== OUTCOMES (not gates) ===')
print('  axiom_v10 false positive rate :', round(float(fp_rate.get("axiom_v10", float("nan"))), 3))
print('  axiom_v03 false positive rate :', round(float(fp_rate.get("axiom_v03", float("nan"))), 3))
print('  baseline  false positive rate :', round(float(fp_rate.get("baseline", float("nan"))), 3))
print('  sensitivity retention v03/v10 :', round(retention, 3))
print('  adversarial restraint v03     :', round(float(adv_restraint.get("axiom_v03", float("nan"))), 2))
print('  primary p (Fisher, clustered) :', round(p_fisher, 5))

print('\n=== VERDICT ===')
if failed:
    verdict = 'NO VERDICT'
    print('NO VERDICT. Failed validity gates:', ', '.join(failed))
    print('The specificity numbers above are descriptive only and cannot be reported as a result.')
elif primary_supported and sensitivity_ok:
    verdict = 'A: revision works'
    print('OUTCOME A. The v0.3 revision reduces the false-positive rate below 0.30 and retains '
          'sensitivity on true-positive cases. The discriminator correction is supported. Report '
          'the effect size, not just the p value, and note that four null cases is a small base.')
elif false_goodhart_ok_v03 and not sensitivity_ok:
    verdict = 'B: refusal, not discrimination'
    print('OUTCOME B. False positives fell but sensitivity fell with them. The revision taught the '
          'loop to decline rather than to discriminate. This is not a fix. Do not report it as one.')
elif not false_goodhart_ok_v03 and fp_rate.get('axiom_v03', 1.0) < fp_rate.get('axiom_v10', 1.0):
    verdict = 'C: partial, still broken'
    print('OUTCOME C. The rate improved but remains above 0.30. The wording changes help and are '
          'not sufficient. Specificity is still too low for the positives to carry information.')
else:
    verdict = 'D: revision does not work'
    print('OUTCOME D. No reduction. The defect is not in the wording of the steps. The honest '
          'conclusion is to narrow the framework to domains where a measured-versus-valued gap is '
          'known in advance to exist, and to stop presenting it as a detector.')

print('\nPre-registration hash :', PREREG_HASH)
print('External witness      :', EXTERNAL_WITNESS)
print('Locked at (UTC)       :', PREREG_TIME)

print('\nUnclosed biases, carried forward and NOT fixed by this run:')
for b in PREREGISTRATION['known_unclosed_biases']:
    print('  -', b)

with open(os.path.join(OUTPUT_DIR, 'verdict_v0.3.json'), 'w') as f:
    json.dump({
        'prereg_sha256': PREREG_HASH,
        'external_witness': EXTERNAL_WITNESS,
        'verdict': verdict,
        'gates': gates,
        'failed_gates': failed,
        'primary_supported': primary_supported,
        'p_fisher': float(p_fisher),
        'false_positive_rate': {k: float(v) for k, v in fp_rate.items()},
        'cause_match_true_positive': {k: float(v) for k, v in cm.items()},
        'sensitivity_retention': float(retention),
        'adversarial_restraint': {k: float(v) for k, v in adv_restraint.items()},
        'max_blinding_condition_recall': float(MAX_RECALL),
        'mean_kappa': None if pd.isna(MEAN_KAPPA) else float(MEAN_KAPPA),
        'known_unclosed_biases': PREREGISTRATION['known_unclosed_biases'],
    }, f, indent=2)
print('\nwrote verdict_v0.3.json')